In [ ]:
import numpy as np

import matplotlib.pyplot as plt


def plot_brachytherapy_dose(
    dose,
    pixel_spacing_mm=(1.0, 1.0),
    isodose_levels=(20, 50, 80, 100),
    source_positions_mm=None,
    dose_unit="Gy",
    title="Illustrativ dosfördelning",
):
    """Visa en 2D-dosmatris med isodoskonturer.

    Args:
        dose: 2D-array med dosvärden.
        pixel_spacing_mm: Pixelstorlek som (x, y) i millimeter.
        isodose_levels: Konturer angivna som procent av maximal dos.
        source_positions_mm: Valfria källpositioner som (x, y) i millimeter.
        dose_unit: Enhet för dosvärdena.
        title: Diagramrubrik.

    Returns:
        (fig, ax): Matplotlib-figur och axel.
    """
    dose = np.asarray(dose, dtype=float)

    if dose.ndim != 2 or dose.size == 0:
        raise ValueError("dose måste vara en icke-tom 2D-array.")
    if not np.isfinite(dose).all():
        raise ValueError("dose får inte innehålla NaN eller oändliga värden.")
    if np.any(dose < 0):
        raise ValueError("dose får inte innehålla negativa värden.")

    spacing_x, spacing_y = pixel_spacing_mm
    if spacing_x <= 0 or spacing_y <= 0:
        raise ValueError("Pixelavstånd måste vara större än noll.")

    max_dose = dose.max()
    if max_dose <= 0:
        raise ValueError("Dosmatrisen måste innehålla minst ett värde över noll.")

    height, width = dose.shape
    extent = (0, width * spacing_x, 0, height * spacing_y)

    fig, ax = plt.subplots(figsize=(8, 6))
    image = ax.imshow(
        dose,
        origin="lower",
        extent=extent,
        cmap="inferno",
        aspect="equal",
    )
    fig.colorbar(image, ax=ax, label=f"Dos ({dose_unit})")

    levels = sorted({
        max_dose * level / 100
        for level in isodose_levels
        if 0 < level <= 100
    })
    if levels:
        contours = ax.contour(
            dose,
            levels=levels,
            origin="lower",
            extent=extent,
            colors="cyan",
            linewidths=1,
        )
        ax.clabel(contours, inline=True, fmt=lambda value: f"{value / max_dose:.0%}")

    if source_positions_mm:
        x, y = zip(*source_positions_mm)
        ax.scatter(x, y, marker="x", color="white", label="Källposition")
        ax.legend()

    ax.set(
        title=title,
        xlabel="x (mm)",
        ylabel="y (mm)",
        xlim=(0, width * spacing_x),
        ylim=(0, height * spacing_y),
    )
    fig.tight_layout()
    return fig, ax